# Sector ETF Portfolio — Regime-Conditional CVaR / DRO

This notebook trades the **11 SPDR Select Sector ETFs** using the regime model trained in `regime_analysis.ipynb`.

| Ticker | Sector |
|--------|--------|
| XLK | Technology |
| XLF | Financials |
| XLE | Energy |
| XLV | Health Care |
| XLI | Industrials |
| XLY | Consumer Discretionary |
| XLP | Consumer Staples |
| XLU | Utilities |
| XLRE | Real Estate (backfilled with VNQ pre-2015) |
| XLB | Materials |
| XLC | Communication Services (backfilled with VOX pre-2018) |

**Prerequisites:** Run `regime_analysis.ipynb` first to generate regime labels and probabilities.

**Data source:** Yahoo Finance via `yfinance` (adjusted close = total return).
Run `pip install yfinance` if not already installed.

In [ ]:
import sys, os, warnings, time
sys.path.insert(0, os.path.abspath('.'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
print('Environment ready. matplotlib backend:', matplotlib.get_backend())

## 1. Download Sector ETF Data

In [ ]:
import importlib, sector_data_loader
importlib.reload(sector_data_loader)
from sector_data_loader import load_sector_returns, SECTOR_TICKERS, SECTOR_LABELS

START_DATE = '2002-01-02'
TEST_START = '2021-01-01'

# Downloads from Yahoo Finance and caches locally in Data/Sector_ETF/
# On subsequent runs it loads from cache (fast)
sector_returns = load_sector_returns(start=START_DATE, cache=True, verbose=True)

# Train / test split
sec_train = sector_returns[sector_returns.index <  TEST_START]
sec_test  = sector_returns[sector_returns.index >= TEST_START]

print(f'\nTrain: {sec_train.index[0].date()} → {sec_train.index[-1].date()}  ({len(sec_train)} days)')
print(f'Test:  {sec_test.index[0].date()}  → {sec_test.index[-1].date()}  ({len(sec_test)} days)')
print(f'\nAnnualised returns (train period):')
print((sec_train.mean() * 252 * 100).round(2).rename(SECTOR_LABELS).to_string())

In [ ]:
# ── Sector return correlations (train period) ─────────────────────────────────
corr = sec_train.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
labels = [SECTOR_LABELS[t] for t in SECTOR_TICKERS]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                fontsize=6, color='white' if abs(corr.values[i,j]) > 0.6 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Sector ETF Return Correlations (Train Period)', fontsize=12)
plt.tight_layout()
fig.savefig('outputs/figures/S01_sector_correlations.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S01_sector_correlations.png')

## 2. Load Regime Model Outputs
These were produced by `regime_analysis.ipynb` — run that notebook first.

In [ ]:
def _load_csv_series(path, col, date_col='Date'):
    return pd.read_csv(path, index_col=date_col, parse_dates=True)[col]

def _load_csv_df(path, date_col='Date'):
    return pd.read_csv(path, index_col=date_col, parse_dates=True)

hard_labels_all  = _load_csv_series('outputs/regime_hard_labels_all.csv',  'regime_int')
regime_probs_all = _load_csv_df('outputs/regime_probabilities_all.csv')
named_labels_df  = _load_csv_df('outputs/regime_named_labels.csv')

# Regime name mapping
regime_names = (
    hard_labels_all.to_frame('regime_int')
    .join(named_labels_df.rename(columns={'regime': 'regime_name'}))
    .dropna().drop_duplicates()
    .set_index('regime_int')['regime_name'].to_dict()
)
regime_names = {int(k): v for k, v in regime_names.items()}

# Split at TEST_START
hard_labels_train = hard_labels_all[hard_labels_all.index <  TEST_START]
hard_labels_test  = hard_labels_all[hard_labels_all.index >= TEST_START]
regime_probs_test = regime_probs_all[regime_probs_all.index >= TEST_START]

print(f'Regime names: {regime_names}')
print(f'Test-set regime distribution:')
for k, label in regime_names.items():
    cnt = (hard_labels_test == k).sum()
    pct = 100 * cnt / len(hard_labels_test)
    print(f'  {label:20s}: {cnt:4d} days ({pct:.1f}%)')

## 3. Sector Returns by Regime
Understanding how each sector behaves in each regime is crucial context before optimising.

In [ ]:
# Align sector train returns with regime labels
common = sec_train.index.intersection(hard_labels_train.index)
sec_aligned = sec_train.loc[common]
lbl_aligned = hard_labels_train.loc[common]

regime_sector_means = {}
regime_sector_vols  = {}

for k, name in regime_names.items():
    mask = lbl_aligned == k
    r    = sec_aligned[mask]
    regime_sector_means[name] = r.mean() * 252 * 100
    regime_sector_vols[name]  = r.std()  * np.sqrt(252) * 100

means_df = pd.DataFrame(regime_sector_means).rename(index=SECTOR_LABELS)
vols_df  = pd.DataFrame(regime_sector_vols).rename(index=SECTOR_LABELS)

print('Annualised sector returns by regime (%):')
print(means_df.round(2).to_string())

means_df.to_csv('outputs/sector_regime_means.csv')
vols_df.to_csv('outputs/sector_regime_vols.csv')
print('\nSaved: outputs/sector_regime_means.csv, outputs/sector_regime_vols.csv')

In [ ]:
# ── Heatmap: sector returns by regime ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, title, fmt in [
    (ax1, means_df, 'Ann. Return (%) by Regime', '{:.1f}'),
    (ax2, vols_df,  'Ann. Vol (%) by Regime',    '{:.1f}'),
]:
    vals = df.values
    if 'Return' in title:
        vmax = max(abs(vals.max()), abs(vals.min()))
        im = ax.imshow(vals, cmap='RdYlGn', vmin=-vmax, vmax=vmax)
    else:
        im = ax.imshow(vals, cmap='YlOrRd', vmin=vals.min(), vmax=vals.max())
    ax.set_xticks(range(len(df.columns)))
    ax.set_yticks(range(len(df.index)))
    ax.set_xticklabels(df.columns, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(df.index, fontsize=9)
    for i in range(len(df.index)):
        for j in range(len(df.columns)):
            ax.text(j, i, fmt.format(vals[i,j]), ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(title, fontsize=11)

fig.suptitle('Sector ETF Behaviour by Regime (Train Period)', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig('outputs/figures/S02_sector_regime_heatmap.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S02_sector_regime_heatmap.png')

## 4. Run Backtest

In [ ]:
import importlib, portfolio_optimiser
importlib.reload(portfolio_optimiser)
from portfolio_optimiser import run_backtest

t0 = time.time()

result = run_backtest(
    returns_test       = sec_test,
    returns_train      = sec_train,
    hard_labels_train  = hard_labels_train,
    hard_labels_test   = hard_labels_test,
    regime_probs_test  = regime_probs_test,
    regime_names       = regime_names,
    # ── CVaR / DRO parameters ───────────────────────────────────────────────
    alpha              = 0.95,
    lam                = 0.005,   # very light reg -> CVaR signal dominates
    gross_limit        = 2.00,    # 200% gross -> more short capacity in Crisis
    dro_kappa          = 1.0,
    # ── Scenario parameters ──────────────────────────────────────────────────
    regime_boost       = 5.0,     # 5x weight on current-regime scenarios
    n_scenarios        = 1000,    # more scenarios -> better tail estimation
    # ── Execution ───────────────────────────────────────────────────────────
    rebalance_freq     = 'D',
    lookback_days      = 756,
    expanding_window   = False,
    verbose            = True,
)

print(f'\nBacktest completed in {time.time()-t0:.1f}s')
print(f'Period: {result.portfolio_returns.index[0].date()} \u2192 {result.portfolio_returns.index[-1].date()}')
result.portfolio_returns.to_csv('outputs/sector_portfolio_returns.csv')
print('Saved: outputs/sector_portfolio_returns.csv')

## 5. Performance Metrics

In [ ]:
print('=' * 90)
print('SECTOR ETF PERFORMANCE METRICS — Test Period (Jan 2021 – present)')
print('=' * 90)
print(result.metrics.T.to_string())

result.metrics.to_csv('outputs/sector_portfolio_metrics.csv')
print('\nSaved: outputs/sector_portfolio_metrics.csv')

## 6. Figures

In [ ]:
COLOURS = {
    'CVaR (Regime)':         '#1a5fa8',
    'DRO (Regime)':          '#1a9e6b',
    'CVaR (Unconditional)':  '#6e9fc9',
    'Equal Weight':          '#aaaaaa',
    'Risk Parity':           '#c84040',
    'MVO':                   '#7c4dcc',
}
REGIME_COLOURS = {
    'Crisis':       '#c0392b',
    'Steady_State': '#27ae60',
    'WOI':          '#e67e22',
    'Inflation':    '#8e44ad',
}

def _regime_bg(ax, regime_labels, dates):
    common = dates.intersection(regime_labels.index)
    rl = regime_labels.loc[common]
    for i in range(len(common) - 1):
        col = REGIME_COLOURS.get(rl.iloc[i], '#dddddd')
        ax.axvspan(common[i], common[i+1], alpha=0.07, color=col, linewidth=0)

print('Colour scheme defined.')

In [ ]:
# ── Figure S3: Cumulative Returns + Drawdown ──────────────────────────────────
pr  = result.portfolio_returns
cum = (1 + pr).cumprod()

fig, (ax_main, ax_dd) = plt.subplots(2, 1, figsize=(14, 8),
                                      gridspec_kw={'height_ratios': [3, 1]},
                                      sharex=True)
for strat, color in COLOURS.items():
    if strat not in cum.columns: continue
    lw = 2.2 if 'Regime' in strat else 1.2
    ls = '-'  if 'Regime' in strat else ('--' if strat in ['60/40','Equal Weight'] else ':')
    ax_main.plot(cum.index, cum[strat], label=strat, color=color, lw=lw, ls=ls)

_regime_bg(ax_main, result.regime_labels, cum.index)
ax_main.set_ylabel('Cumulative Return (log scale)')
ax_main.set_yscale('log')
ax_main.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'{x:.2f}x'))
ax_main.legend(loc='upper left', fontsize=8, ncol=2)
ax_main.set_title('Sector ETF Portfolio — Cumulative Returns (Jan 2021 – Present)', fontsize=12)
ax_main.grid(alpha=0.25, lw=0.5)

for strat, color in COLOURS.items():
    if strat not in cum.columns: continue
    peak = cum[strat].cummax()
    dd   = (cum[strat] - peak) / peak
    lw   = 2.0 if 'Regime' in strat else 1.0
    ax_dd.fill_between(dd.index, dd, 0, alpha=0.2 if 'Regime' in strat else 0.08, color=color)
    ax_dd.plot(dd.index, dd, color=color, lw=lw, ls=('-' if 'Regime' in strat else '--'))

ax_dd.set_ylabel('Drawdown')
ax_dd.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax_dd.grid(alpha=0.25, lw=0.5)
_regime_bg(ax_dd, result.regime_labels, cum.index)

patches = [mpatches.Patch(color=c, alpha=0.4, label=r) for r,c in REGIME_COLOURS.items()]
ax_dd.legend(handles=patches, loc='lower left', fontsize=7, ncol=4,
             title='Regime (background)')

plt.tight_layout()
fig.savefig('outputs/figures/S03_sector_cumulative_returns.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S03_sector_cumulative_returns.png')

In [ ]:
# ── Figure S4: Metrics bar chart ─────────────────────────────────────────────
cvar_col = next((c for c in result.metrics.columns if c.startswith('CVaR')), None)
plot_metrics = ['Sharpe Ratio', 'Sortino Ratio', cvar_col, 'Max Drawdown (%)']
plot_metrics = [m for m in plot_metrics if m and m in result.metrics.columns]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 5))
if len(plot_metrics) == 1: axes = [axes]

for ax, metric in zip(axes, plot_metrics):
    vals   = result.metrics[metric]
    colors = [COLOURS.get(s, '#999') for s in vals.index]
    ax.barh(vals.index, vals.values, color=colors, edgecolor='white', height=0.7)
    ax.set_title(metric, fontsize=10, fontweight='bold')
    ax.axvline(0, color='black', lw=0.8)
    ax.invert_yaxis()
    ax.tick_params(labelsize=8)

fig.suptitle('Sector ETF Strategy Comparison', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig('outputs/figures/S04_sector_metrics.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S04_sector_metrics.png')

In [ ]:
# ── Figure S5: CVaR sector weight evolution ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, strat in zip(axes, ['CVaR (Regime)', 'DRO (Regime)']):
    if strat not in result.weights_history:
        ax.text(0.5, 0.5, f'No data for {strat}', transform=ax.transAxes, ha='center')
        continue
    wh = result.weights_history[strat]
    # Split positive (long) and negative (short) for stacked bar
    wh_long  = wh.clip(lower=0)
    wh_short = wh.clip(upper=0)
    colors   = plt.cm.tab20.colors
    labels   = [SECTOR_LABELS.get(c, c) for c in wh.columns]

    bottom_l = np.zeros(len(wh))
    bottom_s = np.zeros(len(wh))
    for j, (col, label) in enumerate(zip(wh.columns, labels)):
        lv = wh_long[col].values
        sv = wh_short[col].values
        ax.bar(wh.index, lv, bottom=bottom_l, color=colors[j % 20],
               label=label, width=2, linewidth=0)
        ax.bar(wh.index, sv, bottom=bottom_s, color=colors[j % 20],
               width=2, linewidth=0)
        bottom_l += lv
        bottom_s += sv

    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Weight')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set_title(f'{strat} — Sector Weights (long above 0, short below)', fontsize=11)
    ax.legend(loc='upper left', fontsize=6, ncol=6)
    ax.grid(alpha=0.2, lw=0.5, axis='y')
    _regime_bg(ax, result.regime_labels, wh.index)

plt.tight_layout()
fig.savefig('outputs/figures/S05_sector_weights.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S05_sector_weights.png')

In [ ]:
# ── Figure S6: Average sector weights per regime ─────────────────────────────
regimes = sorted(result.regime_labels.dropna().unique())
fig, axes = plt.subplots(2, len(regimes),
                          figsize=(4.5 * len(regimes), 8),
                          squeeze=False)

for si, strat in enumerate(['CVaR (Regime)', 'DRO (Regime)']):
    if strat not in result.weights_history:
        continue
    wh = result.weights_history[strat]
    wh_aligned = wh.reindex(result.regime_labels.index, method='ffill').dropna()
    rl_aligned = result.regime_labels.reindex(wh_aligned.index).dropna()
    wh_aligned = wh_aligned.loc[rl_aligned.index]

    for ri, regime in enumerate(regimes):
        ax   = axes[si][ri]
        mask = rl_aligned == regime
        if mask.sum() == 0:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            continue
        avg_w    = wh_aligned[mask].mean()
        avg_long  = avg_w[avg_w >= 0.005]
        avg_short = avg_w[avg_w < -0.005]

        # Bar chart of long/short weights
        all_nonzero = avg_w[np.abs(avg_w) > 0.005].sort_values()
        colors = ['#c0392b' if v < 0 else '#2980b9' for v in all_nonzero.values]
        ax.barh([SECTOR_LABELS.get(t,t) for t in all_nonzero.index],
                all_nonzero.values * 100, color=colors, edgecolor='white')
        ax.axvline(0, color='black', lw=0.8)
        ax.set_xlabel('Weight (%)')
        rc = REGIME_COLOURS.get(regime, '#888')
        ax.set_title(f'{strat.split(" ")[0]}\n{regime}', fontsize=9,
                     color=rc, fontweight='bold')
        ax.tick_params(labelsize=7)

fig.suptitle('Average Sector Allocation by Regime (blue=long, red=short)', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig('outputs/figures/S06_sector_regime_allocations.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S06_sector_regime_allocations.png')

In [ ]:
# ── Figure S7: Rolling 6-month Sharpe ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
roll = 126
for strat, color in COLOURS.items():
    if strat not in pr.columns: continue
    rs = (pr[strat].rolling(roll).mean() * 252 /
          (pr[strat].rolling(roll).std() * np.sqrt(252) + 1e-8))
    lw = 2.0 if 'Regime' in strat else 1.0
    ls = '-'  if 'Regime' in strat else ('--' if strat in ['60/40','Equal Weight'] else ':')
    ax.plot(rs.index, rs, label=strat, color=color, lw=lw, ls=ls)

ax.axhline(0, color='black', lw=0.8)
_regime_bg(ax, result.regime_labels, pr.index)
ax.set_ylabel('Rolling Sharpe (6-month)')
ax.set_title('Rolling 6-Month Sharpe — Sector ETF Strategies', fontsize=12)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.grid(alpha=0.25, lw=0.5)
ax.set_ylim(-4, 4)
plt.tight_layout()
fig.savefig('outputs/figures/S07_sector_rolling_sharpe.png', dpi=150, bbox_inches='tight')
try:
    from IPython.display import display as _d; _d(fig)
except Exception: plt.show()
print('Saved: outputs/figures/S07_sector_rolling_sharpe.png')

In [ ]:
# ── Regime-conditional Sharpe ─────────────────────────────────────────────────
regime_sharpe_cols = [c for c in result.metrics.columns if c.startswith('Sharpe (')]
if regime_sharpe_cols:
    rs_df = result.metrics[regime_sharpe_cols].T
    rs_df.index = [c.replace('Sharpe (','').rstrip(')') for c in rs_df.index]

    fig, ax = plt.subplots(figsize=(12, 5))
    n_s = len(rs_df.columns)
    x   = np.arange(len(rs_df.index))
    w   = 0.8 / n_s
    for i, strat in enumerate(rs_df.columns):
        ax.bar(x + (i - n_s/2 + 0.5)*w, rs_df[strat].values, w*0.9,
               color=COLOURS.get(strat,'#999'), label=strat, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(rs_df.index, fontsize=10)
    ax.axhline(0, color='black', lw=0.8)
    for i, r in enumerate(rs_df.index):
        ax.axvspan(i-0.5, i+0.5, alpha=0.07,
                   color=REGIME_COLOURS.get(r,'#ddd'), zorder=0)
    ax.set_ylabel('Sharpe Ratio')
    ax.set_title('Regime-Conditional Sharpe — Sector ETF Strategies', fontsize=12)
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig('outputs/figures/S08_sector_regime_sharpe.png', dpi=150, bbox_inches='tight')
    try:
        from IPython.display import display as _d; _d(fig)
    except Exception: plt.show()
    print('Saved: outputs/figures/S08_sector_regime_sharpe.png')

In [ ]:
# ── Save weights history ──────────────────────────────────────────────────────
for strat, wh in result.weights_history.items():
    safe = strat.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    wh.to_csv(f'outputs/sector_weights_{safe}.csv')
    print(f'Saved: outputs/sector_weights_{safe}.csv')

print('\n=== SUMMARY ===')
primary = ['Ann. Return (%)', 'Ann. Vol (%)', 'Sharpe Ratio', 'Sortino Ratio']
cvar_c  = [c for c in result.metrics.columns if c.startswith('CVaR')]
tail    = cvar_c + ['Max Drawdown (%)', 'Calmar Ratio']
print('\n--- Return & Risk ---')
print(result.metrics[[c for c in primary if c in result.metrics.columns]].to_string())
print('\n--- Tail Risk ---')
print(result.metrics[[c for c in tail if c in result.metrics.columns]].to_string())